# AMMS 302 — Week 13: Finding Insights from Big Data (Part 2)
**OMOP CDM analytics — person / visit / condition / measurement + concept vocabulary**

> เปิดคู่กับ [สไลด์ wk13](./wk13.html) — Lab: query OMOP example database ที่สังเคราะห์จาก Synthea

### 🎯 Learning objectives (CLO2/CLO4)
- เขียน SQL บน OMOP schema (JOIN person↔condition↔measurement) ได้
- ใช้ concept/concept_ancestor ค้น cohort เบาหวานได้
- สรุป prevalence by age/gender + baseline table แบบ research

### 📚 Official references
- **CDM v5.4 spec:** [CommonDataModel](https://ohdsi.github.io/CommonDataModel/cdm54.html) — [person](https://ohdsi.github.io/CommonDataModel/cdm54.html#person), [visit_occurrence](https://ohdsi.github.io/CommonDataModel/cdm54.html#visit_occurrence), [condition_occurrence](https://ohdsi.github.io/CommonDataModel/cdm54.html#condition_occurrence), [measurement](https://ohdsi.github.io/CommonDataModel/cdm54.html#measurement)
- **Book of OHDSI:** [ch.5 Standardised data](https://ohdsi.github.io/TheBookOfOhdsi/StandardizedDataStructures.html) · [ch.12 SQL](https://ohdsi.github.io/TheBookOfOhdsi/SqlAndR.html)
- **ATLAS demo:** https://atlas-demo.ohdsi.org/ (design cohorts visually)
- **pysynthea package (OMOP example DB):** [PyPI pysynthea](https://pypi.org/project/pysynthea/) — `setup_db()` โหลด prebuilt OMOP DB จาก Synthea (ต้อง Python ≥3.13!)

### ⚠️ Setup path
```powershell
# ต้องการ python ≥3.13 สำหรับ package pysynthea:
uv python install 3.13
uv venv --python 3.13 ; uv add pysynthea duckdb sqlalchemy matplotlib
```
---


In [ ]:
# §0 Path A (แนะนำ): ใช้ package pysynthea โหลด prebuilt OMOP db
import sys, pathlib
print("python:", sys.version.split()[0])
USE_PYSYNTHEA = False
try:
    from pysynthea.setup.setup import setup_db, connect_db   # type: ignore
    USE_PYSYNTHEA = True
    print("✅ pysynthea available")
except Exception as e:
    print("⚠️ pysynthea unavailable:", str(e)[:120])
    print("→ Path B: ใช้ omop_mini.db จาก week11 notebook แทน")

## §1 โหลด OMOP database (สไลด์ 04–05)
**A)** pysynthea: `setup_db()` download full DB (Zenodo) หรือ `setup_db(database="small")` สร้างเวอร์ชันย่อ  
**B)** fallback: `omop_mini.db` ที่เราสร้างเองใน week11


In [ ]:
if USE_PYSYNTHEA:
    # A: full (~GBs) หรือ small
    setup_db(database="small")
    conn = connect_db(database="small")
    print("connected via pysynthea ✅")
else:
    import duckdb
    p = pathlib.Path("omop_mini.db")
    if not p.exists():
        raise SystemExit("รัน week11-data-standards.ipynb §4 ก่อน เพื่อสร้าง omop_mini.db")
    conn = duckdb.connect(str(p))
    print("connected to local mini-OMOP (duckdb) ✅")

# ตรวจ tables
if USE_PYSYNTHEA:
    import pandas as pd
    tbls = pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema='main'", conn)
    print(tbls.head(15).to_string(index=False))
else:
    print(conn.execute("SHOW TABLES").fetchall())

## §2 Explore core tables (สไลด์ 06)
person · visit_occurrence · condition_occurrence — fields ตาม [spec](https://ohdsi.github.io/CommonDataModel/cdm54.html)


In [ ]:
def q(sql):
    """run sql → pandas (รองรับทั้ง sqlalchemy conn และ duckdb)"""
    try:
        return __import__('pandas').read_sql(sql, conn)
    except Exception:
        return conn.execute(sql).df()

try:
    display(q("SELECT person_id, gender_concept_id, year_of_birth FROM person LIMIT 5"))
    display(q("""SELECT COUNT(*) n FROM person""").rename(columns={'n':'persons'}))
except Exception as e:
    # mini-omop: schema ย่อของเรา
    display(q("SELECT * FROM person LIMIT 5"))
    print("(mini-OMOP mode)")

## §3 Diabetes cohort — condition-based (สไลด์ 07)
OMOP way: ค้นผ่าน `concept` table (SNOMED/RxNorm mapped) — T2DM SNOMED ≈ 195967054 · ICD10CM E11 → concept_id 201820


In [ ]:
# §3.1 ถ้ามี full CDM (pysynthea): JOIN concept
if USE_PYSYNTHEA:
    dm = q("""
      SELECT c.concept_name AS dx, COUNT(*) n
      FROM condition_occurrence co
      JOIN concept c ON c.concept_id = co.condition_concept_id
      WHERE LOWER(c.concept_name) LIKE '%diabetes%'
         OR co.condition_concept_id IN (201820)
      GROUP BY dx ORDER BY n DESC LIMIT 10""")
    display(dm)
else:
    # mini-OMOP: condition_concept_name เป็น text อยู่แล้ว
    dm = q("""
      SELECT condition_concept_name AS dx, COUNT(*) n,
             ROUND(AVG(p.year_of_birth),0) avg_yob
      FROM condition_occurrence c JOIN person p USING(person_id)
      WHERE condition_concept_name LIKE '%Diabetes%'
      GROUP BY dx ORDER BY n DESC""")
    display(dm)

## §4 Cohort demographics + prevalence (สไลด์ 08)


In [ ]:
# §4 DM prevalence by gender (+age decade ถ้ามี year_of_birth)
if USE_PYSYNTHEA:
    prev = q("""
      SELECT CASE p.gender_concept_id WHEN 8507 THEN 'M' WHEN 8532 THEN 'F' ELSE '?' END g,
             COUNT(DISTINCT p.person_id) n_all,
             COUNT(DISTINCT co.person_id) n_dm,
             ROUND(100.0*COUNT(DISTINCT co.person_id)/COUNT(DISTINCT p.person_id),1) pct_dm
      FROM person p
      LEFT JOIN condition_occurrence co
             ON co.person_id=p.person_id
            AND co.condition_concept_id IN (201820)   -- E11/T2DM mapped
      GROUP BY g ORDER BY g""")
else:
    prev = q("""
      SELECT p.gender_concept g,
             COUNT(*) n_all,
             SUM(CASE WHEN c.person_id IS NOT NULL THEN 1 ELSE 0 END) n_dm
      FROM person p
      LEFT JOIN condition_occurrence c
             ON c.person_id=p.person_id AND c.condition_concept_name LIKE '%Diabetes%'
      GROUP BY g""")
display(prev)
print('💡 pct_dm ~30-40% ใน synthetic เพราะ generator ฝัง module ไว้ — ไม่ใช่ค่า population จริง')

## §5 Baseline characteristics table (research style, สไลด์ 09)
Table-1 มาตรฐาน paper: N, age mean±sd, %female, comorbidity counts


In [ ]:
if USE_PYSYNTHEA:
    t1 = q("""
      SELECT COUNT(*) AS N,
             ROUND(AVG(YEAR(CURRENT_DATE)-p.year_of_birth),1) AS age_mean,
             ROUND(100.0*SUM(CASE WHEN gender_concept_id=8532 THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_female
      FROM person p""")
else:
    t1 = q("""SELECT COUNT(*) N,
                     ROUND(AVG(year_of_birth),0) avg_yob,
                     ROUND(100.0*SUM(CASE WHEN gender_concept='FEMALE' THEN 1 ELSE 0 END)/COUNT(*),1) pct_female
              FROM person""")
display(t1.T.rename(columns={0:'value'}))
print('→ ใน paper จริง ตารางนี้มี rows ต่อ covariate; ATLAS สร้างให้อัตโนมัติ (https://atlas-demo.ohdsi.org/)')

In [ ]:
# Cleanup
try: conn.close()
except Exception: pass
print("closed ✅")

### ✅ Self-check
- connect สำเร็จ (path A หรือ B)
- diabetes cohort query ได้ n>0 + อธิบาย concept join ได้
- prevalence table + Table-1 style summary แสดงครบ

### 📝 Homework 13
เลือกโรคอื่น (HTN/Asthma) → หา concept ใน ATLAS demo → เขียน cohort query + prevalence ส่ง .ipynb (อ้าง concept_id ที่ใช้)

---
### 🔗 Specs
[CDM v5.4](https://ohdsi.github.io/CommonDataModel/cdm54.html) · [Book of OHDSI](https://ohdsi.github.io/TheBookOfOhdsi/) · [ATLAS](https://atlas-demo.ohdsi.org/) · [pysynthea PyPI](https://pypi.org/project/pysynthea/)
